# Experimente

API-Anbindung und Durchführung der Experimente. Voraussetzung: `llm_client.py` mit `ask_openai`, `ask_google` und `frage` liegt im selben ORdner; `.env` enthält `OPEN_API_KEY` und `GOOGLE_API_KEY`.

_Kostenregel_: Mechanik immer mit Mini-Prompts testen. Vollen Datensatz (~115.000 Tokens) nur im finalen, geplanten Lauf.


## 1. Setup


In [1]:
import os, json, datetime
from pathlib import Path
from dotenv import load_dotenv

# beim Entwickeln praktisch: llm_client.py wird bei Änderung neu geladen
%load_ext autoreload
%autoreload 2

load_dotenv() # Look for a .env file in the same directory as the Python script


True

In [2]:
# Prüfen, dass beide Keys geladen sind 

print(os.environ.get('OPENAI_API_KEY', 'NICHT GEFUNDEN')[:7])
print(os.environ.get('GOOGLE_API_KEY', 'NICHT GEFUNDEN')[:7])

sk-proj
AQ.Ab8R


## 2. Modelle und System-Prompts


In [3]:
MODEL_OPENAI = 'gpt-5.6-luna'
MODEL_GOOGLE = 'gemini-3.6-flash'

MODELS = [('openai', MODEL_OPENAI), ('google', MODEL_GOOGLE)]

# neutraler System-Prompt, englisch
SYSTEM_PROMPT      = "You are a data analyst answering questions about a dataset provided to you. Answer factually and precisely."
SYSTEM_PROMPT_EXP0 = "You are a data analyst answering questions about exploratory data analysis. Answer factually and precisely."

ITERATIONS = 3

## 3. Funktionen


In [ ]:
from llm_client import ask_openai, ask_google

def ask(provider, prompt, system_prompt=None, temperature=1.0, max_tokens=4000, model=None):

    kwargs = {'model' : model} if model else {}
    if provider == 'openai':
        return ask_openai(prompt, system_prompt=system_prompt, temperature=temperature, max_completion_tokens=max_tokens, **kwargs)
    elif provider == 'google':
        return ask_google(prompt, system_prompt=system_prompt, temperature=temperature, max_tokens=max_tokens, **kwargs)
    else:
        raise ValueError(f"Unbekannter Anbieter: {provider}")

"""
Baut den User Prompt aus einer Frage. Bei with_dataset=True wird der Datensatz vorangestellt. Das ist wichtig für das Caching: gleichbleibender Teil zuerst, wechselnde Frage zuletzt.
"""
def build_prompt(question_text, with_dataset=False, data_text=None):
    if with_dataset:
        if data_text is None:
            raise ValueError("with_dataset=True, but no data_text provided")
        return (
            "Here is a dataset in CSV format:\n\n"
            f"{data_text}\n\n"
            f"Question: {question_text}"
        )
    else:
        return question_text

"""
Hängt einen answer-entry als JSON-Zeile an die JSONL-Datei an. Schützt vor Datenverlust beim Absturz (sofortiges Schreiben)
"""
def save_answer(file, entry):
    file = Path(file)
    file.parent.mkdir(parents=True, exist_ok=True)
    with open(file, 'a', encoding='utf-8') as f:
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

"""Outputfile lesen und Return von question_id, provider und iteration, die schon eine erfolgreiche Antwort haben"""

def already_done(output_file):

    output_file = Path(output_file)
    done = set()
    if not output_file.exists():
        return done
    with open(output_file, encoding='utf-8') as f:
        for line in f:
            try:
                e = json.loads(line)
            except json.JSONDecodeError:
                continue
            # nur die entries zählen, die wirklich eine Antwort haben
            if e.get('answer') is not None:
                done.add((e.get('question_id'), e.get('provider'), e.get('iteration')))

    return done


"""
Führt ein Experiment durch: jede Frage x jedes model x Wiederholungen.
Jede Antwort wird sofort gespeichert. Frischer Kontext pro Aufruf.
"""
def runner(experiment, questions, models, iterations, system_prompt, with_dataset=False, data_text=None, output_file=None):
    if output_file is None:
        output_file = Path('results') / f'{experiment}_answers.jsonl'

    done = already_done(output_file)
    total = len(questions) * len(models) * iterations
    count = 0

    for q in questions:
        user_prompt = build_prompt(q['question'],
                                   with_dataset=with_dataset,
                                   data_text=data_text)
        for provider, model in models:
            for it in range(1, iterations + 1):
                count += 1
                if (q['id'], provider, it) in done:
                    print(f"[{count}/{total}] {q['id']} {provider} it{it}: skip (bereits vorhanden)")
                    continue
                try:
                    r = ask(provider, user_prompt, system_prompt=system_prompt, model=model)

                    entry = {
                        'experiment': experiment,
                        'question_id': q['id'],
                        'category': q.get('category'),
                        'iteration': it,
                        'provider': provider,
                        'model': r['model_requested'],
                        'model_version': r['model_version'],
                        'prompt': user_prompt if not with_dataset else '[dataset + question]',
                        'answer': r['answer'],
                        'input_tokens': r['input_tokens'],
                        'output_tokens': r['output_tokens'],
                        'temperature': r['temperature'],
                        'finish_reason': r['finish_reason'],
                        'timestamp': r['timestamp'],
                    }
                    save_answer(output_file, entry)
                    status = 'ok'
                except Exception as e:
                    save_answer(output_file, {
                        'experiment': experiment,
                        'question_id': q['id'],
                        'iteration': it,
                        'provider': provider,
                        'model': model,
                        'error': str(e),
                    })
                    status = f'ERROR: {e}'

                print(f"[{count}/{total}] {q['id']} {provider} it{it}: {status}")
    print(f"\nFinished. {count} requests, saved in {output_file}")

"""
Lade den Fragenkatalog
"""    
def load_questions(experiment):
    path = Path('fragenkataloge') / f'{experiment}.json'
    with open(path, encoding='utf-8') as f:
        return json.load(f)

## 4. Verbindungstest


In [5]:
for provider, model in MODELS:
    print(f'=== {provider} ===')
    try:
        r = ask(provider, 'Antworte mit genau einem Wort: funktioniert.', model=model)
        print('answer:', r['antwort'])
        print('version:', r['model_version'])
        print('tokens:', r['input_tokens'], '/',
            r['output_tokens'])
    except Exception as e:
        print('Fehler:', e)
    print()

=== openai ===
Fehler: 'antwort'

=== google ===
Fehler: 'antwort'



## 5. Tests (ohne API)

Reiner String-Bau und Speicher-Test ohne Kosten


## 5. Tests ohne API


In [6]:
# Prompt-Bau: E0 (ohne Datensatz) und E1 (mit Mini-Datensatz)
print("=== E0 (ohne Datensatz) ===")
print(build_prompt("What is the IQR method used for?", with_dataset=False))
print()

mini = "Age,MonthlyIncome,Attrition\n35,5000,No\n42,3000,Yes\n28,4500,No"
print("=== E1 (mit Mini-Datensatz) ===")
print(build_prompt("How many rows does the dataset contain?", with_dataset=True, data_text=mini))

=== E0 (ohne Datensatz) ===
What is the IQR method used for?

=== E1 (mit Mini-Datensatz) ===
Here is a dataset in CSV format:

Age,MonthlyIncome,Attrition
35,5000,No
42,3000,Yes
28,4500,No

Question: How many rows does the dataset contain?


## 6. Datensatz Laden

Rohdatensatz als CSV-Text für die datenbehafteten Experimente (E1-E3): Prompts mit diesem Text kosten ~115.000 Input-Token pro Aufruf.


In [7]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path('data') / 'ibm_original.csv')
data_as_text = df.to_csv(index=False)

print('Zeilen', len(df), '| Spalten:', df.shape[1])

Zeilen 1470 | Spalten: 35


## 7. Experiment 0 - kompletter Lauf

EDA-Konzeptwissen, **ohne** Datensatz (billig). 15 Fragen × 2 Modelle × 3 Wiederholungen = 90 Aufrufe.
Wiederaufnahme ist an: ein Neustart überspringt bereits Beantwortetes.


In [10]:
questions_e0 = load_questions('experiment0')

runner(
    experiment='E0',
    questions=questions_e0,
    models=MODELS,
    iterations=ITERATIONS, #3
    system_prompt=SYSTEM_PROMPT_EXP0,
    with_dataset=False,
    output_file=Path('results') / 'E0_answers.jsonl'
)

[1/90] E0_01 openai it1: skip (bereits vorhanden)
[2/90] E0_01 openai it2: skip (bereits vorhanden)
[3/90] E0_01 openai it3: skip (bereits vorhanden)
[4/90] E0_01 google it1: skip (bereits vorhanden)
[5/90] E0_01 google it2: skip (bereits vorhanden)
[6/90] E0_01 google it3: skip (bereits vorhanden)
[7/90] E0_02 openai it1: skip (bereits vorhanden)
[8/90] E0_02 openai it2: skip (bereits vorhanden)
[9/90] E0_02 openai it3: skip (bereits vorhanden)
[10/90] E0_02 google it1: skip (bereits vorhanden)
[11/90] E0_02 google it2: skip (bereits vorhanden)
[12/90] E0_02 google it3: skip (bereits vorhanden)
[13/90] E0_03 openai it1: skip (bereits vorhanden)
[14/90] E0_03 openai it2: skip (bereits vorhanden)
[15/90] E0_03 openai it3: skip (bereits vorhanden)
[16/90] E0_03 google it1: skip (bereits vorhanden)
[17/90] E0_03 google it2: skip (bereits vorhanden)
[18/90] E0_03 google it3: skip (bereits vorhanden)
[19/90] E0_04 openai it1: skip (bereits vorhanden)
[20/90] E0_04 openai it2: skip (bereits 

### Antworten sichten


In [9]:
import json
from collections import Counter

path = "results/e0_answers.jsonl"
rows = [json.loads(l) for l in open(path, encoding="utf-8")]

print("Zeilen gesamt:", len(rows))
answered = Counter((r.get("question_id"), r.get("provider"), r.get("iteration"))
                   for r in rows if r.get("answer") is not None)
print("eindeutige beantwortete Kombinationen:", len(answered))
print("Duplikate:", [k for k, c in answered.items() if c > 1])

# E0_08 google im Detail
for r in rows:
    if r.get("question_id") == "E0_08" and r.get("provider") == "google":
        print("it", r.get("iteration"),
              "ANSWER" if r.get("answer") is not None else "ERROR",
              r.get("timestamp", ""))

Zeilen gesamt: 90
eindeutige beantwortete Kombinationen: 90
Duplikate: []
it 1 ANSWER 2026-08-12 08:35:35
it 2 ANSWER 2026-08-12 08:35:43
it 3 ANSWER 2026-08-14 20:23:06


In [9]:
with open(Path('results') / 'E0_answers.jsonl', encoding='utf-8') as f:
    for line in f:
        e = json.loads(line)
        ans = e.get('answer') or f"[error: {e.get('error')}]"
        print(f"=== {e.get('question_id')} | {e.get('model')} | it{e.get('iteration')} ===")
        print(ans[:300])
        print(f"[finish: {e.get('finish_reason')} | out_tokens: {e.get('output_tokens')}]\n")

=== E0_01 | gpt-5.6-luna | it1 ===
Exploratory data analysis (EDA) is usually an iterative process, but it commonly follows this order:

1. **Define the objective**
   - Clarify the question, business problem, or hypothesis.
   - Identify the outcome or variables of interest.

2. **Collect and understand the data**
   - Load the data
[finish: stop | out_tokens: 516]

=== E0_01 | gpt-5.6-luna | it2 ===
A typical exploratory data analysis (EDA) follows this sequence, although it is often iterative:

1. **Define the objective and questions**  
   Clarify what you want to learn, the relevant variables, and any hypotheses or business questions.

2. **Obtain and understand the data**  
   Identify the 
[finish: stop | out_tokens: 500]

=== E0_01 | gpt-5.6-luna | it3 ===
Exploratory data analysis (EDA) is usually iterative, but a typical sequence is:

1. **Define the objective and questions**
   - Clarify what you want to learn, the unit of analysis, and relevant outcomes or variables.

2. **